In [2]:
import json
from pathlib import Path

import pandas as pd
import statsmodels.api as sm

data = pd.read_csv(
    "data/processed/epl_enhanced_features.csv",
    parse_dates=["Date"]
)

feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
    "elo_difference",
]

X_all = sm.add_constant(data[feature_columns])

# Train on every eligible historical match now that evaluation is complete
final_home_model = sm.GLM(
    data["FTHG"],
    X_all,
    family=sm.families.Poisson()
).fit()

final_away_model = sm.GLM(
    data["FTAG"],
    X_all,
    family=sm.families.Poisson()
).fit()

# Save both models and the information needed to use them later
models_folder = Path("models")

final_home_model.save(models_folder / "poisson_elo_home_goals.pickle")
final_away_model.save(models_folder / "poisson_elo_away_goals.pickle")

metadata = {
    "feature_columns": feature_columns,
    "training_rows": len(data),
    "latest_training_match": str(data["Date"].max().date()),
}

with open(models_folder / "model_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Final models saved successfully.")
print(f"Matches used for final training: {len(data)}")
print(f"Latest match in training data: {data['Date'].max().date()}")

Final models saved successfully.
Matches used for final training: 4106
Latest match in training data: 2026-09-20
